# SVAMITVA Ensemble AI — DGX Training Pipeline

This notebook implements the production training pipeline for the SVAMITVA Feature Extraction Ensemble (V3).

### Staged Training Strategy:
1. **Phase 1: Head Adaptation** — Encoders are frozen. Only the specialized segmentation heads (DeepLabV3+, U-Net++, etc.) are trained.
2. **Phase 2: Global Fine-Tuning** — Encoders are unfrozen with a reduced learning rate (0.1x) to preserve ImageNet features while adapting to drone imagery domain.

**Target Accuracy:** ≥95% IoU.

In [ ]:
import os
import torch
import logging
from pathlib import Path

# ── Environment Setup ──
logging.basicConfig(level=logging.INFO)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
from data.dataset import create_dataloaders
from models.model import EnsembleSvamitvaModel
from models.losses import MultiTaskLoss
from training.trainer import Trainer
from training.config import TrainingConfig

# ── Configuration ──
config = TrainingConfig(
    train_dirs=["data/MAP1"], # Add more MAP folders here
    batch_size=8,
    num_epochs=50,
    learning_rate=3e-4,
    mixed_precision=True,
    freeze_backbone_epochs=5, # Stage 1 length
    checkpoint_dir="check/dgx_run_v1"
)

# ── Data Loaders ──
train_loader, val_loader = create_dataloaders(
    train_dirs=config.train_dirs,
    batch_size=config.batch_size,
    num_workers=4,
    val_split=0.15
)

In [ ]:
# ── Model Initialization ──
model = EnsembleSvamitvaModel(num_roof_classes=5, pretrained=True)
loss_fn = MultiTaskLoss()

# ── Trainer Setup ──
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    config=config
)

print("Training Pipeline Ready.")

## Start Staged Training

The `trainer.fit()` method handles the `freeze_backbone_epochs` logic internally:
- It calls `model.freeze_backbone()` for the first 5 epochs.
- It calls `model.unfreeze_backbone()` thereafter.

In [ ]:
try:
    trainer.fit()
except Exception as e:
    print(f"Training aborted: {e}")
    print("Latest progress saved to check/dgx_run_v1/best_latest.pt")

## Checkpointing Summary

- **best.pt**: The highest-performing weights (based on Avg IoU).
- **best_latest.pt**: The latest saved state (epoch, optimizer, metrics) for crash recovery.
- **training_history.json**: Full loss/metric curves for post-training analysis.